# Fish Farming / Model Training / Image Collector

# Dependencies installation

---


In [ ]:
!pip install eodag rasterio opencv-python

# Variables configuration

---



In [ ]:
from datetime import datetime, timedelta
import torch
import os

#Path to read shape file, the parent folder should contain all related aditional files to .shp, ex. .prj, .dbf, .cpg
path_shp = "/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/external/reference_cages/reference_cages.shp"

#Folder to write the CSV files with coordinates and their relation to polygons
path_shp_out = "/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/reference_cages_out/"

#Folder to download images
download_folder = "/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/raw/sentinel2_rgb"

# Credentials to connect with API for getting images
client_id = "sh-xxxxxxxxxxxxxxxxxxxxxxxxx"
client_secret = "xxxxxxxxxxxxxxxxxx"
token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

# Rank of dates when to get the images (YYYY, MM, D)
start_date = "2024-01-01T00:00:00Z"
end_date = "2024-12-30T00:00:00Z"

#Creating folders if doesnt exist
os.makedirs(path_shp_out, exist_ok=True)

* In order to get credentials for the copernicus dataspace API, it is necesary to create an account in: https://dataspace.copernicus.eu/.

* Aditionally, there should be created the OAuth client. More info in: https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/UserGuides/BeginnersGuide.html#curl-cli-and-python

# Getting coordinates from Shapefiles

---



In [ ]:
import geopandas as gpd
from sklearn.cluster import DBSCAN
import numpy as np
import csv
import hashlib

# Function to hash geometry WKT
def geometry_hash(geom):
    wkt = geom.wkt
    return hashlib.sha256(wkt.encode('utf-8')).hexdigest()

# Load shapefile
gdf = gpd.read_file(path_shp)

# Check if CRS is already set
if gdf.crs is None:
    print("CRS missing, assuming WGS84")
    gdf.set_crs("EPSG:32718", allow_override=True, inplace=True) #projection based on .prj file

# Reproject to coordinate system
gdf = gdf.to_crs(epsg=4326)


# Add geometry hash as unique polygon ID
gdf["geom_hash"] = gdf.geometry.apply(geometry_hash)

# Compute centroids
gdf["centroid"] = gdf.geometry.centroid
coords = np.array([[pt.x, pt.y] for pt in gdf["centroid"]])

# Cluster centroids using DBSCAN (eps in degrees, min_samples=1 to include all)
db = DBSCAN(eps=0.01, min_samples=1).fit(coords) # ~1 km radius
gdf["cluster"] = db.labels_

# Get one representative point per cluster (e.g., the first one)
cluster_centroids = gdf.groupby("cluster")["centroid"].first().reset_index()
coordinates = [(pt.x, pt.y) for pt in cluster_centroids["centroid"]]

print(f"Reduced from {len(coords)} to {len(coordinates)} centroids.")

# Save coordinates with cluster ID to CSV
output_file = path_shp_out + "clustered_centroids.csv"
with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["cluster_id", "longitude", "latitude"])  # Header
    for cluster_id, pt in zip(cluster_centroids["cluster"], cluster_centroids["centroid"]):
        writer.writerow([cluster_id, pt.x, pt.y])

print(f"Saved {len(coordinates)} clustered centroids to {output_file}")

# Save mapping of polygons (using geometry hash) to cluster ID
mapping_file = path_shp_out + "polygon_cluster_mapping.csv"
with open(mapping_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["polygon_from_geometry_hash_id", "cluster_id"])
    for _, row in gdf.iterrows():
        polygon_id = row["geom_hash"]
        cluster_id = row["cluster"]
        writer.writerow([polygon_id, cluster_id])

print(f"Saved polygon-cluster mapping to {mapping_file}")

# Creating session for Getting images

---



In [6]:
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

# Create an OAuth2 session
client = BackendApplicationClient(client_id=client_id)
oauth = OAuth2Session(client=client)

# Get authentication token
def refresh_token(oauth):
  oauth.token = oauth.fetch_token(token_url=token_url, client_secret=client_secret, include_client_id=True)
  print(" Token received:", oauth.token["access_token"])

#to get error from server side
def sentinelhub_compliance_hook(response):
    response.raise_for_status()
    return response

oauth.register_compliance_hook("access_token_response", sentinelhub_compliance_hook)

# Getting images

---



In [7]:
import os
import json
import math
from eodag import EODataAccessGateway
from requests.exceptions import RequestException, HTTPError
from oauthlib.oauth2 import BackendApplicationClient, TokenExpiredError
from requests_oauthlib import OAuth2Session
from datetime import datetime, timedelta
import csv
from io import BytesIO
import rasterio
import numpy as np


# Define headers with authentication token
refresh_token(oauth)
headers = {"Authorization": f"Bearer {oauth.token['access_token']}", "Accept": "image/tiff"}

# Define Sentinel-2 RGB Extraction Script
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B02", "B03", "B04"],
    output: { bands: 3},
  }
}

function evaluatePixel(sample) {
  // Multiply by 5 to enhance brightness and contrast
  return [5 * sample.B04, 5 * sample.B03, 5 * sample.B02]
}
"""


# Coordinates transformed into bounding boxes
def create_bbox(id, lon, lat):
    # Rough approximation: 1° ≈ 111,000 m
    offset_deg = 0.01 # ~2.2km x 2.2km
    return {
        "cluster_id": id,
        "lonmin": round(lon - offset_deg, 6),
        "latmin": round(lat - offset_deg, 6),
        "lonmax": round(lon + offset_deg, 6),
        "latmax": round(lat + offset_deg, 6)
    }

# Read coordinates from CSV file
coordinates = []
with open(path_shp_out + "clustered_centroids.csv", mode="r") as file:
    reader = csv.DictReader(file)
    for row in reader:
        cluster_id = int(row["cluster_id"])
        lon = float(row["longitude"])
        lat = float(row["latitude"])
        coordinates.append((cluster_id, lon, lat))

bounding_boxes = [create_bbox(cluster_id, lon, lat) for cluster_id, lon, lat in coordinates]


# Request RGB Sentinel-2 Images
os.makedirs(download_folder, exist_ok=True)

for bbox in bounding_boxes:
      request = {
          "input": {
              "bounds": {
                  "properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"},
                  "bbox":[
                            bbox['lonmin'], bbox['latmin'],
                            bbox['lonmax'], bbox['latmax']
                        ]
              },
              "data": [{
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                          "timeRange": {
                            "from": start_date,
                            "to": end_date
                          },
                          "maxCloudCoverage": 20
                        },
                        "processing": {
                          "mosaicking": "leastCC"
                        }
                      }]
          },
          "output": {
                    "width": 512,
                    "height": 512,

                    "responses": [{
                      "identifier": "default",
                      "format": {
                        "type": "image/tiff"
                      }
                    }]
                  },
          "evalscript": evalscript
      }

      url = "https://sh.dataspace.copernicus.eu/api/v1/process"


      try:
          response = oauth.post(url, json=request, headers=headers)
          response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
          #print(f"Successfully retrieved image for cluster {bbox['cluster_id']}")

      except TokenExpiredError:
          print("Token expired. Refreshing...")
          refresh_token(oauth)  # call oauth.refresh_token(...)
          response = oauth.post(url, json=request, headers=headers)  # retry
          response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
          print(f"Successfully retrieved image for cluster {bbox['cluster_id']} after token refresh")
      except HTTPError as e:
          print(f"HTTP error retrieving image for cluster {bbox['cluster_id']}: {e}")
          if e.response is not None:
              print(f"Response status code: {e.response.status_code}")
              print(f"Response content: {e.response.text}")
          continue # Continue to the next bounding box if an error occurs
      except RequestException as e:
          print(f"Error retrieving image for cluster {bbox['cluster_id']}: {e}")
          continue # Continue to the next bounding box if an error occurs


      # Check if image is black before saving
      with rasterio.open(BytesIO(response.content)) as src:
          arr = src.read()
          if np.all(arr == 0):
              print(f"Cluster {bbox['cluster_id']} discarded → black image for lon-lat {bbox['lonmin']}-{bbox['latmin']}")
              continue

      # Save if not black
      file_name = f"sentinel2_{bbox['cluster_id']}.tiff"
      file_path = os.path.join(download_folder, file_name)
      with open(file_path, "wb") as file:
          file.write(response.content)
      print(f"Image saved: {file_path}")

 Token received: eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJYVUh3VWZKaHVDVWo0X3k4ZF8xM0hxWXBYMFdwdDd2anhob2FPLUxzREZFIn0.eyJleHAiOjE3NTkzNDI2NDksImlhdCI6MTc1OTM0MjA0OSwianRpIjoiZWZmYzUyZWQtYjUxMS00MTA1LWE1YjctM2RjNTg3ZmYyZjJmIiwiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5kYXRhc3BhY2UuY29wZXJuaWN1cy5ldS9hdXRoL3JlYWxtcy9DRFNFIiwic3ViIjoiM2VmMmNkNTktZTViMC00NTc4LTkyYmQtODQzZmJlOWMwODJjIiwidHlwIjoiQmVhcmVyIiwiYXpwIjoic2gtOTFiZDdlZjEtZmRiNS00ZDNkLWE3MGEtYzgwOTM5NWFmODc2Iiwic2NvcGUiOiJlbWFpbCBwcm9maWxlIHVzZXItY29udGV4dCIsImVtYWlsX3ZlcmlmaWVkIjpmYWxzZSwiY2xpZW50SG9zdCI6IjM1LjE5Ni4yNS4xNDUiLCJvcmdhbml6YXRpb25zIjpbImRlZmF1bHQtMGQzNDUxZWUtNWI4Yy00NWZjLWE0ZGQtZTNkZGQ2ODI5ZmQ1Il0sInVzZXJfY29udGV4dF9pZCI6IjY5M2I2MWJiLTRlOGMtNDEzNy1hYjk3LTMzOThkYjVhODBjMCIsImNvbnRleHRfcm9sZXMiOnt9LCJjb250ZXh0X2dyb3VwcyI6WyIvYWNjZXNzX2dyb3Vwcy91c2VyX3R5cG9sb2d5L2NvcGVybmljdXNfZ2VuZXJhbC8iLCIvb3JnYW5pemF0aW9ucy9kZWZhdWx0LTBkMzQ1MWVlLTViOGMtNDVmYy1hNGRkLWUzZGRkNjgyOWZkNS8iXSwicHJlZmVycmVkX3VzZXJuYW1lIjoic2VydmljZS1hY2NvdW5